# Penmanshiel WT08 Derivation

This notebook records the decision log behind the `Penmanshiel_Hourly_WT08` benchmark dataset. It is derived from the Penmanshiel WT08 raw SCADA data and keeps an hourly single-turbine series for forecasting robustness experiments.

The executable preprocessing contract lives in `scripts/preprocess_penmanshiel_hourly_wt08.py`. This notebook summarizes the rationale and checks behind the shipped benchmark dataset.

Original source: Plumley and Takeuchi, 2025, Penmanshiel wind farm data, Zenodo v3, https://zenodo.org/records/16807304
Source subset: `Penmanshiel_SCADA_2016_WT01-10_3107.zip` through `Penmanshiel_SCADA_2022_WT01-10_4462.zip`. WT08 is in the WT01-10 files. Later 2023 and 2024 files in v3 are not part of the frozen benchmark slice.
Benchmark derived dataset: `Penmanshiel_Hourly_WT08`


## EDA Summary

1. Inspect the raw 2016-2022 SCADA coverage and discover which turbines are available in the shared study window.
2. Apply one preprocessing order to every turbine: split on Power (kW) gaps longer than 1 day, inspect kept-segment feature gaps, then drop long-gap features.
3. Select the turbine and window with the longest retained forecastable hourly history under that shared policy.
4. Remove leakage-style cumulative, availability, and controller expectation columns before deriving the final feature set.
5. Forward fill within the selected 10-minute segment, aggregate complete target hours by mean, and verify the final hourly Parquet file.

No backfill is used anywhere in the exported series. Benchmark window defaults stay in `configs/dataset_windows.yaml`, not in this notebook.


## Slice Rationale

The all-turbine screen used the same preprocessing order for every turbine discovered in the 2016-2022 coverage window. WT08 is the continuity leader with 25,867 retained hourly rows from 2016-08-18 15:00:00 to 2019-08-01 09:00:00.

Important alternatives from the turbine screen:

| Turbine | Role in screen | Result |
| --- | --- | --- |
| WT08 | Selected continuity leader | 25,867 retained hourly rows with 65 modeled channels, corr=0.9798 and MAE=76.37 to the site-median Power (kW) series |
| WT10 | Strongest representativeness candidate | corr=0.9854 and MAE=63.45 to the site-median Power (kW) series, but only 19,496 retained hourly rows |
| WT09 | Channel-richness leader | 213 modeled channels, but only 19,515 retained hourly rows |

WT08 is selected because no other turbine keeps more retained history under the same preprocessing order while staying close to the site-median Power (kW) series. Its site-median correlation is 0.9798, only 0.0056 below WT10, and its retained segment is 6,371 hourly rows longer than WT10. Extra-channel turbines are materially shorter and later.

## Window and Channel Contract

| Check | Benchmark contract |
| --- | --- |
| Raw WT08 range | 2016-07-27 15:40:00 to 2022-12-31 23:50:00 |
| Selected 10-minute segment | 2016-08-18 13:30:00 to 2019-08-01 09:00:00 |
| Exported hourly window | 2016-08-18 15:00:00 to 2019-08-01 09:00:00 |
| Exported rows | 25,867 hourly rows |
| Stored columns | `datetime` plus 65 modeled channels |
| Continuous channels | 65 SCADA variables |
| Discrete channels | none |
| Target alias | `power` |
| Repository key | `Penmanshiel_Hourly_WT08` |
| Expected file | `data/processed/penmanshiel_hourly_wt08.parquet` |

Power (kW) gaps longer than 1 day create target boundaries before feature selection or filling. Multi-day target outages are removed first, then the WT08 kept-segment feature-gap distribution is inspected to decide which channels are safe to keep.


## Feature-Gap Decision

After leakage pruning, WT08 has 249 numeric channels before kept-segment gap dropping. The 21-day feature-gap rule falls inside a clean empirical gap in the selected WT08 segment:

| Gap check | Value |
| --- | --- |
| Largest max gap among kept channels | 0.8194 days |
| Smallest max gap among dropped channels | 35.5764 days |
| Empty interval around the active cutoff | 34.7569 days |
| Modeled channels retained | 65 |
| Long-gap channels dropped | 184 |


## Preprocessing Contract

- Load WT08 raw SCADA records for the shared 2016-2022 coverage window.
- Remove leakage-style cumulative, availability, contractual, power-curve, and controller expectation columns.
- Split first on Power (kW) gaps longer than 1 day.
- Select the longest remaining target-continuous 10-minute segment.
- Drop non-target channels whose kept-segment max gap is longer than 21 days.
- Forward fill only within the selected segment.
- Trim unresolved leading rows if needed after forward fill.
- Aggregate complete target hours by the mean of six 10-minute values.
- Write rows in strictly increasing hourly timestamp order.


## Benchmark Validation

Prepare from raw data with `uv run python scripts/preprocess_penmanshiel_hourly_wt08.py --raw-source <path-to-penmanshiel-scada-directory-or-zip> --output data/processed/penmanshiel_hourly_wt08.parquet`.

Validate the staged benchmark file with `uv run python scripts/preprocess_penmanshiel_hourly_wt08.py --output data/processed/penmanshiel_hourly_wt08.parquet --validate-existing`.

The validation path checks the registry contract, row count, column order, hourly datetime column, channel typing, and finite modeled values.
